# Mejora 1: más épocas con *warmup* + *cosine annealing* del learning rate

**Faster R-CNN v2 (ResNet-50 + FPN) sobre *pills-sxdht*.** Primer experimento de mejora sobre el baseline (`baseline-faster-r-cnn-v2.ipynb`).

## Hipótesis

El análisis de errores del baseline (sección 13) concluye que **el margen principal está en la localización**: a IoU 0.85 el error `loc` cuesta +0.096 de AP en validación y +0.092 en test, mientras que a IoU 0.5 casi no queda margen (mAP@0.5 = 0.981 en test). El modelo encuentra y clasifica bien las pastillas, pero **no ajusta las cajas al píxel**.

Dos síntomas del entrenamiento del baseline apuntan al *learning rate*:

1. **El mAP de validación seguía subiendo en la última época** (0.881 en la época 15, la mejor): el entrenamiento se cortó antes de converger.
2. **Con LR constante el mAP de validación oscila** entre épocas (0.852 → 0.819 → 0.861, 0.878 → 0.860 → 0.881). Con un paso de 0.005 fijo, el optimizador sigue "rebotando" alrededor del mínimo y no puede hacer los ajustes pequeños que necesita la regresión de cajas para pasar de IoU 0.8 a 0.9.

Una política que **baje el LR progresivamente hasta casi cero** debería permitir ese ajuste fino al final del entrenamiento, y más épocas dan tiempo a que ocurra.

## Qué cambia respecto al baseline

| Aspecto | Baseline | Mejora 1 |
|---|---|---|
| Épocas | 15 | **30** |
| Política de LR | constante (0.005) | ***warmup* lineal 1 época + *cosine annealing* hasta 0**, actualizado por iteración |
| Todo lo demás | | **igual**: datos, limpieza, modelo, SGD (lr máx. 0.005, momentum 0.9, wd 5e-4), batch 4, semilla, evaluación |

- ***Warmup***: durante la primera época el LR sube linealmente desde 0.001·LR hasta LR. Evita que los gradientes grandes de la cabeza nueva (inicializada al azar) desajusten los pesos preentrenados de COCO en las primeras iteraciones. Es lo que hace el script de referencia de torchvision para detección.
- ***Cosine annealing***: después el LR baja siguiendo medio coseno, de 0.005 a 0. Pasa mucho tiempo con LR alto (exploración) y se frena suavemente al final (ajuste fino), sin los saltos bruscos de un `StepLR`.

Las dos cosas forman una única estrategia: **la política de entrenamiento** (cuánto se entrena y con qué paso). Si hay tiempo, se puede separar el efecto de cada parte repitiendo este cuaderno con `NUM_EPOCHS = 15` (mismo presupuesto que el baseline, solo cambia el *scheduler*).

## Cómo se comprueba

- **Métrica de decisión**: mAP@0.5:0.95 en **validación** (test solo se usa para informar).
- **Comprobación de la hipótesis**: si la mejora actúa sobre la localización, debería bajar el ΔAP de `loc` a IoU 0.85 en la descomposición TIDE (sección 6), y subir sobre todo el mAP@0.5:0.95 y no tanto el mAP@0.5.
- **Coste**: el modelo es el mismo, así que el tiempo de inferencia no debe cambiar; el de entrenamiento se duplica (~80 min en una T4).

## 1. Configuración

El código común con el baseline (dataset, limpieza de clases, DataLoaders, entrenamiento, evaluación y TIDE) está en `common.py`. Si no está junto al cuaderno (Kaggle, Colab), se descarga del repositorio del grupo.

Todos los hiperparámetros están en esta celda. Los marcados con `# CAMBIO` son los únicos que difieren del baseline.

In [ ]:
# Dependencias que pueden no estar instaladas
try:
    import datasets, torchmetrics, pycocotools  # noqa: F401
except ImportError:
    %pip install -q datasets torchmetrics pycocotools

import sys
import urllib.request
from pathlib import Path

if not Path("common.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HectorBarro/CV1_P6/main/common.py", "common.py")
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
import torch

import common as C

%matplotlib inline

In [ ]:
EXP_NAME = "mejora01_cosine_fasterrcnn"
DESCRIPTION = "30 épocas con warmup lineal (1 época) + cosine annealing del LR, por iteración"

# ---------------- Hiperparámetros (iguales al baseline salvo los marcados) ----------------
SEED = 42
NUM_EPOCHS = 30          # CAMBIO: baseline 15
BATCH_SIZE = 4
LR = 0.005               # LR máximo (tras el warmup)
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS = 1        # CAMBIO: baseline sin warmup
WARMUP_FACTOR = 0.001    # el LR arranca en LR * WARMUP_FACTOR
MIN_LR_FACTOR = 0.0      # el coseno termina en LR * MIN_LR_FACTOR

OUT_DIR = C.RESULTS_ROOT / EXP_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

C.set_seed(SEED)
print("Dispositivo:", C.describe_device())

## 2. Datos y modelo

Exactamente igual que en el baseline (secciones 2 a 6): mismas particiones, misma limpieza de clases (7 clases + fondo, se eliminan las imágenes con `Ibuphil 600 mg`), sin aumento de datos, y Faster R-CNN v2 con pesos `COCO_V1` y un predictor nuevo.

Se mantiene también el **orden de las llamadas que usan la semilla** (modelo, DataLoaders, lote de ejemplo), para que la inicialización de la cabeza y el orden de las imágenes coincidan con los del baseline. Así la diferencia de resultados se debe a la política de LR y no al azar. En GPU puede quedar una pequeña variación porque algunas operaciones de torchvision no son deterministas.

In [ ]:
hf, CLASSES = C.load_splits()
NUM_CLASSES = len(CLASSES)

model = C.build_model(NUM_CLASSES, seed=SEED)

ds = C.make_datasets(hf)
loaders = C.make_loaders(ds, BATCH_SIZE, SEED)
print(f"Imágenes -> train: {len(ds['train'])} | valid: {len(ds['valid'])} | test: {len(ds['test'])}")

# Igual que en el baseline: se saca un lote de ejemplo (esto también consume el generador de train)
images, targets = next(iter(loaders["train"]))
print("Ejemplo:", images[0].shape, "| cajas:", targets[0]["boxes"].shape[0], "| etiquetas:", targets[0]["labels"].tolist())

## 3. Optimizador y política de learning rate

El optimizador es el mismo SGD del baseline. El *scheduler* se actualiza **en cada iteración** (no en cada época), así que el coseno es una curva suave y el *warmup* ocupa las primeras `len(train_loader)` iteraciones.

La gráfica muestra el LR que se va a usar en cada iteración, frente al LR constante del baseline.

In [ ]:
C.set_seed(SEED)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

iters_per_epoch = len(loaders["train"])
total_iters = NUM_EPOCHS * iters_per_epoch
warmup_iters = WARMUP_EPOCHS * iters_per_epoch
scheduler = C.warmup_cosine_scheduler(optimizer, total_iters, warmup_iters,
                                      warmup_factor=WARMUP_FACTOR, min_factor=MIN_LR_FACTOR)

# Vista previa del LR (con un optimizador de prueba, para no tocar el real)
_opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=LR)
_sch = C.warmup_cosine_scheduler(_opt, total_iters, warmup_iters, WARMUP_FACTOR, MIN_LR_FACTOR)
lrs = []
for _ in range(total_iters):
    lrs.append(_opt.param_groups[0]["lr"])
    _opt.step()
    _sch.step()

epochs_axis = np.arange(total_iters) / iters_per_epoch
plt.figure(figsize=(8, 3.2))
plt.plot(epochs_axis, lrs, label="mejora 1: warmup + cosine")
plt.plot([0, 15], [LR, LR], "--", c="gray", label="baseline: constante (15 épocas)")
plt.xlabel("época")
plt.ylabel("learning rate")
plt.title(f"Política de LR ({iters_per_epoch} iteraciones por época)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "politica_lr.png", dpi=120)
plt.show()

## 4. Entrenamiento

Mismo bucle que el baseline: en cada época se registran las pérdidas de train y validación y el mAP de validación, y se guarda el modelo con **mayor mAP@0.5:0.95 en validación**. Test no se usa para ninguna decisión.

> ~160 s por época en una Tesla T4 → unos 80 min en total.

In [ ]:
C.set_seed(SEED)
fit_result = C.fit(model, loaders, optimizer, NUM_EPOCHS, OUT_DIR, scheduler=scheduler)
history, best_epoch = fit_result["history"], fit_result["best_epoch"]

## 5. Curvas de entrenamiento frente al baseline

Los resultados del baseline se leen de `resultados/baseline_fasterrcnn/` (o se descargan del repositorio si no están en local).

Qué mirar:
- **mAP de validación**: con LR constante oscilaba; con el coseno debería estabilizarse y seguir subiendo cuando el LR se hace pequeño (últimas épocas).
- **Pérdida de validación**: si vuelve a subir mientras la de train baja, el entrenamiento largo estaría sobreajustando.

In [ ]:
baseline = C.load_experiment("baseline_fasterrcnn")
baseline["history"].setdefault("lr", [baseline["results"]["config"]["lr"]] * len(baseline["history"]["val_map"]))

C.plot_history_comparison({"baseline": baseline["history"], "mejora 1 (cosine, 30 ép.)": history},
                          out_path=OUT_DIR / "curvas_vs_baseline.png")

b_hist = baseline["history"]["val_map"]
print(f"val mAP baseline : mejor {max(b_hist):.4f} (época {int(np.argmax(b_hist)) + 1}) | "
      f"desv. típica últimas 5 épocas {np.std(b_hist[-5:]):.4f}")
print(f"val mAP mejora 1 : mejor {max(history['val_map']):.4f} (época {best_epoch}) | "
      f"desv. típica últimas 5 épocas {np.std(history['val_map'][-5:]):.4f}")

## 6. Evaluación en test

Con el mejor modelo de validación, igual que en la sección 10 del baseline: mAP de COCO, AP por clase, curvas PR y tiempo de inferencia (batch 1, mismo hardware).

In [ ]:
res, per_class = C.test_report(model, loaders["test"], CLASSES)
C.plot_pr_curves(res, CLASSES, out_path=OUT_DIR / "curvas_pr_test.png")

times_ms = C.measure_inference(model, ds["test"])
print(f"\nHardware: {C.hardware_name()} | inferencia: {times_ms.mean():.1f} ± {times_ms.std():.1f} ms/imagen "
      f"-> {1000 / times_ms.mean():.1f} FPS")

### 6.1 Comparación con el baseline

Entre paréntesis, la diferencia con el baseline. La decisión de quedarse o no con la mejora se toma con la primera fila (**val mAP@0.5:0.95**); las filas de test son la estimación independiente del rendimiento.

Con 45 imágenes de validación y 89 de test, diferencias de **menos de 1 punto** de mAP pueden deberse al azar del entrenamiento: hay que interpretarlas con cautela.

In [ ]:
config = {"seed": SEED, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
          "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY, "optimizer": "SGD",
          "scheduler": f"warmup lineal {WARMUP_EPOCHS} época (x{WARMUP_FACTOR}) + cosine hasta x{MIN_LR_FACTOR}, por iteración",
          "augmentation": None}
summary = C.make_summary(EXP_NAME, DESCRIPTION, config, fit_result, res, per_class, times_ms)

C.compare_metrics({"baseline": baseline["results"], "mejora 1": summary})

## 7. ¿Ha mejorado la localización? (TIDE)

Se repite la descomposición TIDE del baseline (sección 13.3) a IoU 0.5 y 0.85. Si la hipótesis es correcta, **el ΔAP de `loc` a IoU 0.85 debe bajar** respecto al baseline (+0.096 en validación, +0.092 en test), mientras que el resto de errores no tiene por qué cambiar.

También se guardan las predicciones de validación y test (`preds_valid.json`, `preds_test.json`): servirán para la comparativa final y para probar TTA o ensembles sin volver a ejecutar el modelo.

In [ ]:
split_data = {
    "valid": C.collect_predictions(model, loaders["valid"], CLASSES),
    "test": C.collect_predictions(model, loaders["test"], CLASSES),
}
C.save_predictions(split_data, OUT_DIR)

tide = C.tide_analysis(split_data)
print()
C.compare_tide({"baseline": baseline["tide"], "mejora 1": C.tide_to_json(tide)})

In [ ]:
# AP en función del umbral de IoU (sección 13.2 del baseline): si mejora la localización,
# la curva debe caer más tarde que la del baseline
mv = C.import_mapviz()
for name, (gts, preds) in split_data.items():
    mv.plot_ap_vs_iou(gts, {f"mejora 1 ({name})": preds})

## 8. Guardar resultados

`results.json`, `history.json`, `tide.json` y las predicciones quedan en `resultados/mejora01_cosine_fasterrcnn/`. Los `.json` se suben al repositorio para el cuaderno de comparativa; los pesos (`best_model.pth`) y las figuras no.

In [ ]:
C.save_experiment(OUT_DIR, summary, tide)

# En Kaggle: comprimir la carpeta de resultados para descargarla
if Path("/kaggle/working").exists():
    import shutil
    from IPython.display import FileLink

    shutil.make_archive(f"/kaggle/working/{EXP_NAME}", "zip", OUT_DIR)
    display(FileLink(f"{EXP_NAME}.zip"))

## 9. Conclusiones

> **Completar tras ejecutar el cuaderno**, con los números de las secciones 5, 6.1 y 7:
>
> - ¿Sube el mAP@0.5:0.95 de **validación** respecto al baseline (0.881)? ¿Y en test (0.840)?
> - ¿La mejora se concentra en mAP@0.5:0.95 / mAP@0.75 y no en mAP@0.5? (coherente con una mejora de localización)
> - ¿Baja el ΔAP de `loc` a IoU 0.85? ¿Cambian los otros tipos de error (`miss`, `cls`)?
> - ¿Se estabiliza el mAP de validación al final del entrenamiento? ¿Hay sobreajuste en la pérdida de validación?
> - Coste: tiempo de entrenamiento (x2) y de inferencia (debería ser igual).
>
> **Decisión**: si mejora en validación, esta política de LR pasa a ser la receta de entrenamiento de las siguientes mejoras (resolución, aumento de datos...). Tiene sentido además porque el aumento de datos suele necesitar más épocas para dar resultado.